In [14]:
import re
import shutil
import argparse
import hls4ml
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from hls4ml.converters import convert_from_pytorch_model
from brevitas.nn import QuantConv2d
from brevitas.nn import QuantLinear
from brevitas.nn import QuantReLU

In [3]:
with open('teste.yaml', "r") as f:
    config = yaml.safe_load(f)

#config
model_name = config['ModelName']
model = f'{model_name}.keras'

In [12]:
import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(4, 20, kernel_size=3, bias=False)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv2d(20, 8, kernel_size=1, bias=False)
        self.relu2 = nn.ReLU()

        self.fc1 = nn.Linear(2048, 6, bias=False)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        return x

model = Net()

In [ ]:
state_dict = torch.load("sat6-cnn-t1w2.pt", map_location="cpu")

missing, unexpected = model.load_state_dict(
    state_dict,
    strict=False
)

print("missing =", missing)
print("unexpected =", unexpected)

missing = []
unexpected = ['relu1.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value', 'relu2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value']


In [16]:
model.summary

AttributeError: 'Net' object has no attribute 'summary'

In [ ]:
hls_model = convert_from_pytorch_model(
    model,
    hls_config=config["HLSConfig"],
    output_dir=config["OutputDir"],
    backend=config["Backend"],
    part=config["Part"],
    clock_period=config["ClockPeriod"],
    io_type=config["IOType"],
    project_name=config["ProjectName"],
)

Interpreting Model ...


TypeError: 'NoneType' object is not iterable

In [ ]:
hls4ml.utils.plot_model(
    hls_model,
    show_shapes=True,
    show_precision=True,
    to_file='output',
)
